In [2]:
import glob, os
from pathlib import Path
from natsort import natsorted

# Fixing mtb/cyto merged masks

## Find masks to regenerate

In [6]:
zarr_addresses = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_13/zarr/*top*.zarr')
zarr_addresses = natsorted([fn for fn in zarr_addresses if 'max_proj' in fn])
zarr_addresses

['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_13/zarr/rep_2_mouse_13_top_max_proj.zarr']

In [7]:
import os
import glob
import json
import zarr
import numpy as np
from tqdm.auto import tqdm
import dask.array as da
import rasterio.features as features
from shapely.geometry import shape
from shapely.validation import make_valid
from rasterio.transform import Affine

# Establish the connection
for zarr_fn in tqdm(zarr_addresses):
    # 1. Locate the construct in the real world
    geojson_dir = os.path.dirname(zarr_fn).replace('zarr', 'qu_path')
    if not os.path.exists(geojson_dir):
        print(f"Skipping {os.path.basename(zarr_fn)}: No QuPath directory found.")
        continue

    # 2. Identify the matching signal
    basename = os.path.basename(zarr_fn)
    orientation = next((tag for tag in ['top', 'bot', 'left', 'right'] if tag in basename), None)
    
    candidates = glob.glob(os.path.join(geojson_dir, '*cyto*.geojson'))
    target_geojson = None

    if orientation:
        target_geojson = next((c for c in candidates if orientation in os.path.basename(c)), None)
    elif candidates:
        # If no orientation tag, avoid files that DO have one, or take the only option
        target_geojson = next((c for c in candidates if not any(tag in os.path.basename(c) for tag in ['top', 'bot', 'left', 'right'])), candidates[0])

    if not target_geojson:
        print(f"Skipping {os.path.basename(zarr_fn)}: No matching GeoJSON found.")
        continue

    print(f"Processing: {os.path.basename(zarr_fn)} -> {os.path.basename(target_geojson)}")

    # 3. Load the Matrix dimensions
    try:
        # Accessing level 0 directly for dimensions
        root = zarr.open(zarr_fn, mode='r+')
        # Assuming typical OME-Zarr structure, image data is usually at '0'
        # If your data is at '0/0' (multiscales), adjust accordingly. 
        # Using dask to peek at shape without loading
        d0 = da.from_zarr(f"{zarr_fn}/s0") 
        shape_yx = d0.shape[-2:] # Last two dimensions (Y, X)
    except Exception as e:
        print(f"  Error reading Zarr dimensions: {e}")
        continue

    # 4. Extract and refine geometries
    with open(target_geojson) as f:
        print('reading geojson')
        gj_data = json.load(f)

    geoms = []

    # Processing the raw feed
    for feat in tqdm(gj_data.get("features", []), desc="Processing Geometries"):
        geom = feat.get("geometry")
        if not geom:
            continue
    
        g = shape(geom)
    
        # Correcting structural integrity
        if not g.is_valid:
            try:
                g = make_valid(g)
            except Exception:
                g = g.buffer(0)
    
        # Discarding noise (Area <= 1000)
        if g.area <= 1000:
            continue
    
        if not g.is_empty:
            geoms.append(g)
    
    # Remove FOV contour if it dominates the signal
    if len(geoms) > 1:
        areas = [g.area for g in geoms]
        if max(areas) > 200000:
            max_idx = int(np.argmax(areas))
            del geoms[max_idx]

    # 5. Materialize Instance Segmentation (Unique IDs)
    # Create iterable of (geometry, value) for rasterize
    # Values start at 1, 0 is background
    shapes_with_ids = [(g, i + 1) for i, g in enumerate(geoms)]

    instance_mask = features.rasterize(
        shapes_with_ids,
        out_shape=shape_yx,
        fill=0,
        dtype="uint32", # Sufficient for < 65,535 cells
        transform=Affine.identity(),
    )

    # 6. Write to Zarr under labels/cyto_seg
    labels_root = root.require_group("labels")
    lbl_group = labels_root.require_group("cyto_seg")

    # Overwrite dataset '0'
    if "0" in lbl_group:
        del lbl_group["0"]

    arr_out = lbl_group.create_dataset(
        "0",
        data=instance_mask,
        shape=instance_mask.shape,
        chunks=(512,512), # Auto-chunking
        dtype="uint32",
        overwrite=True
    )

    # 7. Metadata injection (OME-NGFF)
    lbl_group.attrs["multiscales"] = [{
        "name": "cyto_seg",
        "version": "0.4",
        "axes": [
            {"name": "y", "type": "space", "unit": "pixel"},
            {"name": "x", "type": "space", "unit": "pixel"},
        ],
        "datasets": [{"path": "0"}],
    }]
    # Link label to image source
    lbl_group.attrs["image-label"] = {
        "version": "0.4",
        "source": {"image": "../../"}, 
    }
    
    # Register this label in the root 'labels' list if not present
    if "labels" not in labels_root.attrs:
         labels_root.attrs["labels"] = ["cyto_seg"]
    else:
        current_labels = labels_root.attrs["labels"]
        if "cyto_seg" not in current_labels:
            labels_root.attrs["labels"] = current_labels + ["cyto_seg"]

    print(f"  ✓ Written labels/cyto_seg/0 [{instance_mask.shape}] with {len(geoms)} instances")

print("\nSystem update complete.")

  0%|          | 0/1 [00:00<?, ?it/s]

Processing: rep_2_mouse_13_top_max_proj.zarr -> rep2_Mouse_13_top_max_proj_cyto.geojson
reading geojson


Processing Geometries:   0%|          | 0/156130 [00:00<?, ?it/s]

/tmp/ipykernel_2091536/3912812305.py:110: ZarrDeprecationWarning: Use Group.create_array instead.
  arr_out = lbl_group.create_dataset(


  ✓ Written labels/cyto_seg/0 [(38614, 40020)] with 155909 instances

System update complete.


In [ ]:
from pathlib import Path
import json
from tqdm.auto import tqdm
import numpy as np
import zarr
import dask.array as da
from affine import Affine
from shapely.geometry import shape
from shapely.validation import make_valid
from rasterio import features

base = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice")

for mouse_dir in tqdm(sorted(base.glob("mouse_*")), total=len(sorted(base.glob("mouse_*")))):
    zarr_dir = mouse_dir / "zarr"
    geojson_dir = mouse_dir / "qu_path"

    if not zarr_dir.exists() or not geojson_dir.exists():
        continue

    # Zarrs to process (skip .prev / .jnotebook)
    zarrs = [
        z for z in zarr_dir.glob("*.zarr")
        if not any(s in z.name for s in [".prev", ".jnotebook"])
    ]
    geojsons = list(geojson_dir.glob("*.geojson"))
    if not geojsons:
        continue

    for zarr_fn in zarrs:
        print(f"\n🧩 Processing {zarr_fn.name}")

        # open zarr
        root = zarr.open(str(zarr_fn), mode="a")

        # use level 0/0 as main image
        arr = da.from_zarr(f"{zarr_fn}/0/0")   # (..., Y, X)
        shape_yx = arr.shape[-2:]

        # identity pixel transform (row, col) -> (y, x)
        transform = Affine.identity()

        # one merged mask per Zarr
        merged_mask = np.zeros(shape_yx, dtype="uint8")

        for gj in geojsons:
            print(f"  importing {gj.name}")
            with open(gj) as f:
                gj_data = json.load(f)

            geoms = []
            for feat in gj_data.get("features", []):
                geom = feat.get("geometry")
                if geom:
                    g = shape(geom)
                    if g.area > 2000:  # skip huge polygons (likely FOV contour)
                        continue
                    # fix invalids/self-intersections
                    try:
                        g = make_valid(g)
                    except Exception:
                        g = g.buffer(0)
                    if not g.is_empty:
                        geoms.append(g)

            if not geoms:
                print("   ⚠️ no geometries found, skipping")
                continue

            # --- drop the largest polygon (likely the FOV contour) ---
            if len(geoms) > 1:
                areas = [g.area for g in geoms]
                max_idx = int(np.argmax(areas))
                geoms = [g for i, g in enumerate(geoms) if i != max_idx]

            if not geoms:
                print("   ⚠️ only FOV contour present, skipping")
                continue

            # burn polygons -> binary mask
            mask = features.rasterize(
                [(g, 1) for g in geoms],
                out_shape=shape_yx,
                fill=0,
                dtype="uint8",
                transform=transform,
            )

            # OR into the merged mask
            merged_mask |= mask

        # ---- write as NGFF label: labels/ground_truth_mtb/0 ----
        labels_root = root.require_group("labels")
        lbl_group = labels_root.require_group("ground_truth_mtb")

        # overwrite dataset '0' if it exists
        if "0" in lbl_group:
            del lbl_group["0"]

        arr_out = lbl_group.create_array(
            "0",
            shape=merged_mask.shape,
            dtype="uint8",
            chunks="auto",
        )
        arr_out[...] = merged_mask

        # minimal NGFF label metadata
        lbl_group.attrs["multiscales"] = [{
            "name": "ground_truth_mtb",
            "version": "0.4",
            "axes": [
                {"name": "y", "type": "space", "unit": "pixel"},
                {"name": "x", "type": "space", "unit": "pixel"},
            ],
            "datasets": [{"path": "0"}],
        }]
        lbl_group.attrs["image-label"] = {
            "version": "0.4",
            "source": "..",   # parent = image root
        }

        print(f"   ✓ wrote labels/ground_truth_mtb/0 [{merged_mask.shape}]")

print("\n✅ Done — NGFF labels written as labels/ground_truth_mtb/0 in each .zarr")


### Move manually outputted geojson to correct dir

In [44]:
from pathlib import Path
from tqdm.auto import tqdm
import shutil, os

src_dir = Path('/home/dayn/QuPath/bin')            # where the *.geojson are
dst_root = Path('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice')

fns = sorted(src_dir.glob('*.geojson'))

for fn in tqdm(fns):
    mouse_n = fn.stem                 # 'mouse_1', 'mouse_2', ...
    dest_dir = dst_root / mouse_n / 'qu_path'   # or 'qu_path/arx' if you prefer
    dest_dir.mkdir(parents=True, exist_ok=True)

    dest_file = dest_dir / fn.name    # <-- full file path, not just the directory

    try:
        # use copy (no xattrs) to avoid EACCES on network shares
        shutil.copy(str(fn), str(dest_file))
    except PermissionError:
        # last-ditch fallback: manual copy
        with open(fn, 'rb') as r, open(dest_file, 'wb') as w:
            w.write(r.read())

print("done")


  0%|          | 0/10 [00:00<?, ?it/s]

done


# Convert geojson to zarr labels

In [110]:
from pathlib import Path
import json
from tqdm.auto import tqdm
import numpy as np
import zarr
import dask.array as da
from affine import Affine
from shapely.geometry import shape
from shapely.validation import make_valid
from rasterio import features

base = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice")

for mouse_dir in tqdm(sorted(base.glob("mouse_*")), total=len(sorted(base.glob("mouse_*")))):
    zarr_dir = mouse_dir / "zarr"
    geojson_dir = mouse_dir / "qu_path"

    if not zarr_dir.exists() or not geojson_dir.exists():
        continue

    # Zarrs to process (skip .prev / .jnotebook)
    zarrs = [
        z for z in zarr_dir.glob("*.zarr")
        if not any(s in z.name for s in [".prev", ".jnotebook"])
    ]
    geojsons = list(geojson_dir.glob("*.geojson"))
    if not geojsons:
        continue

    for zarr_fn in zarrs:
        print(f"\n🧩 Processing {zarr_fn.name}")

        # open zarr
        root = zarr.open(str(zarr_fn), mode="a")

        # use level 0/0 as main image
        arr = da.from_zarr(f"{zarr_fn}/0/0")   # (..., Y, X)
        shape_yx = arr.shape[-2:]

        # identity pixel transform (row, col) -> (y, x)
        transform = Affine.identity()

        # one merged mask per Zarr
        merged_mask = np.zeros(shape_yx, dtype="uint8")

        for gj in geojsons:
            print(f"  importing {gj.name}")
            with open(gj) as f:
                gj_data = json.load(f)

            geoms = []
            for feat in gj_data.get("features", []):
                geom = feat.get("geometry")
                if geom:
                    g = shape(geom)
                    if g.area > 2000:  # skip huge polygons (likely FOV contour)
                        continue
                    # fix invalids/self-intersections
                    try:
                        g = make_valid(g)
                    except Exception:
                        g = g.buffer(0)
                    if not g.is_empty:
                        geoms.append(g)

            if not geoms:
                print("   ⚠️ no geometries found, skipping")
                continue

            # --- drop the largest polygon (likely the FOV contour) ---
            if len(geoms) > 1:
                areas = [g.area for g in geoms]
                max_idx = int(np.argmax(areas))
                geoms = [g for i, g in enumerate(geoms) if i != max_idx]

            if not geoms:
                print("   ⚠️ only FOV contour present, skipping")
                continue

            # burn polygons -> binary mask
            mask = features.rasterize(
                [(g, 1) for g in geoms],
                out_shape=shape_yx,
                fill=0,
                dtype="uint8",
                transform=transform,
            )

            # OR into the merged mask
            merged_mask |= mask

        # ---- write as NGFF label: labels/ground_truth_mtb/0 ----
        labels_root = root.require_group("labels")
        lbl_group = labels_root.require_group("ground_truth_mtb")

        # overwrite dataset '0' if it exists
        if "0" in lbl_group:
            del lbl_group["0"]

        arr_out = lbl_group.create_array(
            "0",
            shape=merged_mask.shape,
            dtype="uint8",
            chunks="auto",
        )
        arr_out[...] = merged_mask

        # minimal NGFF label metadata
        lbl_group.attrs["multiscales"] = [{
            "name": "ground_truth_mtb",
            "version": "0.4",
            "axes": [
                {"name": "y", "type": "space", "unit": "pixel"},
                {"name": "x", "type": "space", "unit": "pixel"},
            ],
            "datasets": [{"path": "0"}],
        }]
        lbl_group.attrs["image-label"] = {
            "version": "0.4",
            "source": "..",   # parent = image root
        }

        print(f"   ✓ wrote labels/ground_truth_mtb/0 [{merged_mask.shape}]")

print("\n✅ Done — NGFF labels written as labels/ground_truth_mtb/0 in each .zarr")


  0%|          | 0/11 [00:00<?, ?it/s]


🧩 Processing 20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250814_5375.zarr
  importing mouse_1.geojson
   ✓ wrote labels/ground_truth_mtb/0 [(41702, 58291)]

🧩 Processing 20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250814_5375.zarr
  importing mouse_10.geojson
   ✓ wrote labels/ground_truth_mtb/0 [(41702, 58291)]

🧩 Processing 20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250902_5555.zarr
  importing mouse_11.geojson
   ✓ wrote labels/ground_truth_mtb/0 [(41702, 60365)]

🧩 Processing 20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5551_jnotebook.zarr
  importing mouse_2.geojson
   ✓ wrote labels/ground_truth_mtb/0 [(37555, 49997)]

🧩 Processing 20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5551.zarr
  importing mouse_2.geojson
   ✓ wrote labels/ground_truth_mtb/0 [(37555, 49997)]

🧩 Processing 20250901_40X_Time

## Fixing top/bottom pairs

